# Task 3.3: Trace Logger

Owner: Jessica Bedson. Depends on Task 3.1 (Enock's agent loop).

**Plan definition:** every call, argument and return persisted in replayable form.

This notebook shows the logger working with Enock's `run_agent`, unchanged. The tools are dummy functions and the language model is a scripted stand-in, so no WRDS data and no provider call is involved.

Files:
- `trace_logger.py`
- `test_trace_logger.py`

## What the logger records

| Event | When | Contents |
|---|---|---|
| `run_start` | logger created | format version, run id, metadata (firm alias, year, model version, seed, LLM, call budget, prompt and schema hashes) |
| `tool_call` | every executed call | seq, expected result id, tool, arguments, status, return or error, return SHA-256, timestamps |
| `agent_trace` | end of run | the agent loop's own trace, including rejected calls and budget events |
| `final` | end of run | raw final output (saved before verification), status, calls used |
| `verification` | end of run | verifier report and the verified output |
| `reconcile` | end of run | whether the logged calls match the agent's own trace |
| `run_end` | end of run | counts and a completion flag |

Each line is written and flushed when it happens, copied rather than referenced, never overwritten, and protected by a digest.

## Setup

The first cell finds `trace_logger.py` (in `src/`, next to this notebook, or one folder up) and Enock's `run_agent`. For `run_agent` it tries the repo module first; if that is not importable, it loads the agent-loop cell straight from Enock's notebook (`Task 3.1  3.2. Agent Loop-draft 1.ipynb`) if that file is in this folder or one folder up.

In [13]:
import json
import sys
import types
from pathlib import Path

HERE = Path.cwd()
for candidate in [HERE, HERE / "src", HERE.parent, HERE.parent / "src"]:
    if (candidate / "trace_logger.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from trace_logger import TraceLogger, load_trace, replay_check, text_sha256


def load_agent_loop():
    """Return Enock's agent-loop module: the repo module if importable, else the cell from his notebook."""
    for module_name in ("agent_loop", "src.agent", "agent"):
        try:
            module = __import__(module_name, fromlist=["run_agent"])
            if hasattr(module, "run_agent"):
                print(f"Agent loop: imported from module {module_name!r}")
                return module
        except ImportError:
            pass
    for folder in [HERE, HERE.parent, HERE.parent / "notebooks"]:
        for nb_path in sorted(folder.glob("Task 3.1*Agent Loop*.ipynb")):
            nb = json.loads(nb_path.read_text(encoding="utf-8"))
            for cell in nb["cells"]:
                src = "".join(cell["source"])
                if cell["cell_type"] == "code" and "def run_agent" in src:
                    module = types.ModuleType("agent_loop")
                    sys.modules["agent_loop"] = module      # needed for dataclasses
                    exec(compile(src, str(nb_path), "exec"), module.__dict__)
                    print(f"Agent loop: loaded from notebook {nb_path.name}")
                    return module
    raise ImportError(
        "Could not find Enock's run_agent. Either put his agent module on the path "
        "(and adjust the module names above), or copy 'Task 3.1  3.2. Agent Loop-draft 1.ipynb' "
        "into this folder."
    )


agent_loop = load_agent_loop()
run_agent = agent_loop.run_agent
verify_evidence_links = agent_loop.verify_evidence_links
SYSTEM_PROMPT = agent_loop.SYSTEM_PROMPT
TOOL_SCHEMAS = agent_loop.TOOL_SCHEMAS

TRACE_DIR = Path("traces")

Agent loop: imported from module 'agent_loop'


## Dummy tools and a scripted language model

In [14]:
# Dummy tools only. Real tools come from src/tools.py; the logger wraps them the same way.
def score_firm(gvkey, feature_year):
    return {"firm_id": gvkey, "feature_year": feature_year, "outcome_year": feature_year + 1,
            "probability": 0.0421, "model_version": "dummy_v0", "status": "ok",
            "source": "dummy: no real data"}

def get_control_history(gvkey, through_fyear):
    return {"firm_id": gvkey, "through_fyear": through_fyear, "adverse_years": [2015],
            "source": "dummy: no real data"}

def get_industry_context(sic, feature_year):
    raise LookupError(f"No dummy industry data for SIC {sic}.")

TOOLS = {
    "score_firm": score_firm,
    "get_control_history": get_control_history,
    "get_industry_context": get_industry_context,
}


class ScriptedLLM:
    """Stands in for Gemini / GPT-OSS: returns a fixed sequence of responses."""

    def __init__(self, responses):
        self.responses = list(responses)

    def complete(self, messages, tools):
        return self.responses.pop(0)


final_answer = {
    "assessment": {"risk_level": "moderate", "summary": "Dummy demonstration only.",
                   "firm_id": "FIRM_A", "feature_year": 2018, "outcome_year": 2019,
                   "probability": 0.0421, "model_version": "dummy_v0", "confidence": "low"},
    "evidence": [
        {"claim": "Model probability is 4.2%.", "tool_result_ids": ["result_1"], "interpretation": "Score."},
        {"claim": "Adverse opinion in 2015.", "tool_result_ids": ["result_2"], "interpretation": "History."},
        {"claim": "Industry rate is high.", "tool_result_ids": ["result_9"], "interpretation": "Unsupported."},
    ],
    "contradictory_evidence": [],
    "uncertainty": ["Industry context unavailable."],
    "recommended_checks": [{"check": "Retry industry context.", "reason": "Tool failed."}],
}

llm = ScriptedLLM([
    {"type": "tool_call", "name": "score_firm", "arguments": {"gvkey": "FIRM_A", "feature_year": 2018}},
    {"type": "tool_call", "name": "get_control_history", "arguments": {"gvkey": "FIRM_A", "through_fyear": 2018}},
    {"type": "tool_call", "name": "not_a_tool", "arguments": {}},           # rejected by the loop
    {"type": "tool_call", "name": "get_industry_context", "arguments": {"sic": 2834, "feature_year": 2018}},
    {"type": "final", "content": json.dumps(final_answer)},
])

## Run the agent with logging

In [15]:
logger = TraceLogger(TRACE_DIR, metadata={
    "firm_id": "FIRM_A",
    "feature_year": 2018,
    "model_version": "dummy_v0",
    "llm": "scripted stand-in",
    "max_tool_calls": 8,
    "system_prompt_sha256": text_sha256(SYSTEM_PROMPT),
    "tool_schemas_sha256": text_sha256(json.dumps(TOOL_SCHEMAS, sort_keys=True)),
    "data": "dummy",
})

result = run_agent(
    firm_id="FIRM_A", feature_year=2018, user_question="Assess next-year adverse opinion risk.",
    llm=llm, tools=logger.wrap_tools(TOOLS), max_tool_calls=8,
)
# The logger saves the raw output first, then runs the verifier, then saves the verified output.
summary = logger.finish(agent_result=result, verifier=verify_evidence_links)
print(json.dumps({k: v for k, v in summary.items() if k != "verification"}, indent=2))

{
  "run_id": "20261109T200023589955Z_35ca3889",
  "path": "traces/20261109T200023589955Z_35ca3889.jsonl",
  "status": "completed",
  "tool_calls_logged": 3,
  "tool_errors": 1,
  "reconciled": true,
  "verified_items": 2,
  "blocked_items": 1
}


## Read the trace back and check it

In [16]:
trace = load_trace(summary["path"])
print("Valid:", trace["valid"], "| Complete:", trace["complete"], "| Problems:", trace["problems"])
print("Reconciled with the agent's own trace:", trace["reconcile"]["matches"])
raw_claims = len(trace["final"]["output"]["evidence"])
kept_claims = len(trace["verification"]["verified_output"]["evidence"])
print(f"Evidence claims: {raw_claims} raw, {kept_claims} after verification (raw output preserved)")
print()
for call in trace["tool_calls"]:
    outcome = call["return"] if call["status"] == "ok" else call["error"]
    print(f"{call['seq']}. {call['tool']}({call['arguments']}) -> {call['status']}: {outcome}")

rejected = [e for e in trace["events"] if e["event"] == "agent_trace"][0]["trace"]
print("\nRejected by the loop (kept in agent_trace):",
      [e["tool"] for e in rejected if e.get("event") == "tool_rejected"])

Valid: True | Complete: True | Problems: []
Reconciled with the agent's own trace: True
Evidence claims: 3 raw, 2 after verification (raw output preserved)

1. score_firm({'feature_year': 2018, 'gvkey': 'FIRM_A'}) -> ok: {'feature_year': 2018, 'firm_id': 'FIRM_A', 'model_version': 'dummy_v0', 'outcome_year': 2019, 'probability': 0.0421, 'source': 'dummy: no real data', 'status': 'ok'}
2. get_control_history({'gvkey': 'FIRM_A', 'through_fyear': 2018}) -> ok: {'adverse_years': [2015], 'firm_id': 'FIRM_A', 'source': 'dummy: no real data', 'through_fyear': 2018}
3. get_industry_context({'feature_year': 2018, 'sic': 2834}) -> error: {'message': 'No dummy industry data for SIC 2834.', 'type': 'LookupError'}

Rejected by the loop (kept in agent_trace): ['not_a_tool']


## Notes for the team

- **No change to the agent loop.** The logger wraps the tool functions. Enock's `run_agent` and Mark's `_jsonable` fix work as they are.
- **Result ids.** The logger numbers executed calls `result_1`, `result_2`, ... the same way `run_agent` does, and `reconcile` checks that they agree. If the loop's numbering changes, reconcile flags it.
- **Rejected calls.** Calls the loop rejects (unknown tool, bad arguments) are never executed, so they are not tool calls. They are kept in the `agent_trace` event.
- **Trace viewer (Task 5.4).** Mouzam's viewer can read `traces/*.jsonl` with `load_trace`.
- **Data.** Traces hold tool returns. With real data they contain WRDS-derived values, so keep `traces/` out of git until the data-use question is settled.